# 반도체 공정 스케줄링 시뮬레이션 결과

## 환경 설정

In [1]:
import os
import simpy
from dotenv import load_dotenv

# .env 파일의 환경 변수 로드
load_dotenv()

# .env 파일의 파라미터
SIMUL_TIME = int(os.getenv('SIMUL_TIME', 100))
BASE_DATA_PATH = os.getenv('BASE_DATA_PATH', 'data')

print(f"시뮬레이션 시간: {SIMUL_TIME}")
print(f"데이터 경로: {BASE_DATA_PATH}")

시뮬레이션 시간: 100
데이터 경로: data

## 모듈 import

In [2]:
from simulation import DataLoader, Scheduler, Job, Machine
# 향후 알고리즘 모듈 추가 예정
# from algorithms.genetic import GeneticScheduler
# from algorithms.rule_based import RuleBasedScheduler

## 데이터 로드

In [3]:
data_loader = DataLoader(BASE_DATA_PATH)

data = data_loader.load_all_data()

print("=" * 60)
print("데이터 개요")
print("=" * 60)
print(f"Jobs: {len(data['jobs'])} 개")
print(f"Operations: {len(data['operations'])} 개")
print(f"Machines: {len(data['machines'])} 개")
print(f"Machine Failures: {len(data['machine_failure'])} 개")
print(f"Setup Times: {len(data['setup_times'])} 개")
print(f"Operation-Machine Map: {len(data['operation_machine_map'])} 개")

데이터 개요
Jobs: 10 개
Operations: 35 개
Machines: 8 개
Machine Failures: 8 개
Setup Times: 12 개
Operation-Machine Map: 95 개

## 시뮬레이션 실행

In [4]:
# SimPy 환경 생성
env = simpy.Environment()

# 스케줄러 생성
scheduler = Scheduler(
    env=env,
    machine_df=data['machines'],
    operations_df=data['operations'],
    machine_failure_df=data['machine_failure'],
    setup_times_df=data['setup_times'],
    op_machine_df=data['operation_machine_map']
)

## 작업 생성

In [5]:
jobs = []

for _, job_info in data['jobs'].iterrows():
    # 해당 작업의 operation 정보 가져오기
    job_operations = data['operations'].loc[
        data['operations']['job_id'] == job_info['job_id'],
        ['op_id', 'op_seq', 'qtime']
    ].sort_values('op_seq')
    
    # Job 인스턴스 생성
    job = Job(
        env=env,
        job_info=job_info.to_dict(),
        op_info=job_operations,
        scheduler=scheduler
    )
    jobs.append(job)

## 시뮬레이션 실행

In [6]:
# 간트 차트로 구성 예정
env.run(until=SIMUL_TIME)

0   Job J1 starts setup for operation J1_O1 on machine M1
0   Job J2 starts setup for operation J2_O1 on machine M2
0   Job J1 starts processing operation J1_O1 on machine M1
0   Job J2 starts processing operation J2_O1 on machine M2
5   Job J3 starts setup for operation J3_O1 on machine M3
5   Job J3 starts processing operation J3_O1 on machine M3
10  Job J2 finished operation J2_O1 on machine M2
10  Job J2 starts setup for operation J2_O2 on machine M4
10  Job J4 starts setup for operation J4_O1 on machine M2
10  Job J2 starts processing operation J2_O2 on machine M4
10  Job J4 starts processing operation J4_O1 on machine M2
11  Job J1 finished operation J1_O1 on machine M1
11  Job J1 starts setup for operation J1_O2 on machine M5
11  Job J5 starts setup for operation J5_O1 on machine M1
11  Job J1 starts processing operation J1_O2 on machine M5
11  Job J5 starts processing operation J5_O1 on machine M1
20  Job J3 finished operation J3_O1 on machine M3
20  Job J6 starts setup for ope

## 시뮬레이션 KPI 및 통계

In [7]:
# 향후 결과 분석 기능 추가
# - 작업 완료율
# - 평균 대기 시간
# - 장비 가동률
# - QTime 위반 건수
# - 처리량 (Throughput)